# ECE 175B ADG — Kaggle T4 Training

> Spring 2026 final project. Train Attribute-Disentangled Guidance on CelebA 64×64.
>
> **Time budget**: ~6h on T4 GPU (25 epochs). Kaggle session limit 12h — should fit.
>
> **Dataset**: Must add Kaggle CelebA dataset input before running.
> Go to **Add Data** → search "celeba-dataset" → use jessicali9530/celeba-dataset.

## Cell 1: Install dependencies

In [ ]:
!pip install -q diffusers>=0.27 accelerate>=0.27 tqdm
print("Dependencies installed.")

## Cell 3: Auto-write project .py files (no manual upload needed)

Cell below writes `model.py`, `ddpm.py`, `cfg.py`, `adg.py` to `/kaggle/working/` from inlined source. Just run it.


In [ ]:
# Cell 3: Auto-write 4 project .py files to /kaggle/working/
# (Inlined by build script — no manual upload needed)
import os
from pathlib import Path
os.chdir("/kaggle/working")

INLINED = {
    'model.py': '"""Conditional UNet for CelebA 64×64 — 用 diffusers 的 UNet2DModel 改 attribute 条件。\n\nConditioning 设计:\n- attribute K-hot vector (K=4) 经过一个小 MLP → 时间步 embedding 维度\n- 在 timestep embedding 上加上 attribute embedding (residual style)\n- Null token: 训练时 10% drop attribute (置 K-zeros) 实现 classifier-free training\n"""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nfrom diffusers import UNet2DModel\n\n\nclass AttrConditionedUNet(nn.Module):\n    """在 diffusers 的 UNet2DModel 之上加 attribute conditioning."""\n\n    def __init__(\n        self,\n        sample_size: int = 64,\n        in_channels: int = 3,\n        out_channels: int = 3,\n        n_attrs: int = 4,\n        time_embed_dim: int = 256,\n    ):\n        super().__init__()\n        self.n_attrs = n_attrs\n        self.time_embed_dim = time_embed_dim\n\n        # diffusers UNet2DModel 自带 timestep embedding,\n        # 我们用 class_embed_type="identity" 把 attribute embedding 注入它的 class_emb\n        self.unet = UNet2DModel(\n            sample_size=sample_size,\n            in_channels=in_channels,\n            out_channels=out_channels,\n            layers_per_block=2,\n            block_out_channels=(64, 128, 256, 256),\n            down_block_types=(\n                "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D",\n            ),\n            up_block_types=(\n                "UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D",\n            ),\n            class_embed_type="identity",  # 我们直接传 (B, time_embed_dim) 进去\n        )\n\n        # attribute (K,) → time_embed_dim\n        self.attr_proj = nn.Sequential(\n            nn.Linear(n_attrs, time_embed_dim),\n            nn.SiLU(),\n            nn.Linear(time_embed_dim, time_embed_dim),\n        )\n\n    def forward(self, x: torch.Tensor, t: torch.Tensor, attr: torch.Tensor) -> torch.Tensor:\n        """\n        Args:\n            x: (B, 3, H, W), 噪声图\n            t: (B,), timestep\n            attr: (B, K), float 0/1 — null 用 全 0\n        Returns:\n            (B, 3, H, W) 预测的噪声 ε\n        """\n        cond = self.attr_proj(attr)  # (B, time_embed_dim)\n        return self.unet(x, t, class_labels=cond).sample\n\n\ndef make_null_attr(batch_size: int, n_attrs: int, device: torch.device) -> torch.Tensor:\n    """null token = all-zeros K-hot vector."""\n    return torch.zeros(batch_size, n_attrs, device=device)\n\n\ndef random_attr_dropout(attr: torch.Tensor, p: float = 0.1) -> torch.Tensor:\n    """以概率 p 整批 drop attribute → null. CFG 训练标配."""\n    mask = (torch.rand(attr.shape[0], 1, device=attr.device) < p).float()\n    return attr * (1 - mask)\n\n\nif __name__ == "__main__":\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = AttrConditionedUNet(sample_size=64, n_attrs=4).to(device)\n    n_params = sum(p.numel() for p in model.parameters())\n    print(f"Model parameters: {n_params / 1e6:.2f}M")\n\n    # smoke test\n    B = 4\n    x = torch.randn(B, 3, 64, 64, device=device)\n    t = torch.randint(0, 1000, (B,), device=device)\n    attr = torch.randint(0, 2, (B, 4), device=device).float()\n    with torch.no_grad():\n        out = model(x, t, attr)\n    print(f"Forward OK. Output: {out.shape}")\n',
    'ddpm.py': '"""DDPM training step — 用 diffusers DDPMScheduler.\n\nForward: q(x_t | x_0) = N(√α̅_t x_0, (1-α̅_t) I)\nReverse: parameterized by ε_θ(x_t, t, attr) — model 预测加进去的噪声\n\nLoss = MSE(ε_predicted, ε_true) — proposal §4 / Ho 2020\n"""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn.functional as F\nfrom diffusers import DDPMScheduler\n\n\ndef make_scheduler(num_train_timesteps: int = 1000) -> DDPMScheduler:\n    return DDPMScheduler(\n        num_train_timesteps=num_train_timesteps,\n        beta_schedule="linear",\n        beta_start=0.0001,\n        beta_end=0.02,\n        prediction_type="epsilon",\n    )\n\n\ndef training_step(\n    model,\n    scheduler: DDPMScheduler,\n    batch: dict,\n    device: torch.device,\n    attr_drop_p: float = 0.1,\n) -> torch.Tensor:\n    """单个 batch 的训练 loss.\n\n    Args:\n        model: AttrConditionedUNet\n        batch: {"image": (B,3,H,W) ∈ [-1,1], "attr": (B,K) ∈ {0,1}}\n    Returns:\n        scalar loss\n    """\n    from model import random_attr_dropout\n\n    x0 = batch["image"].to(device)\n    attr = batch["attr"].to(device)\n    attr = random_attr_dropout(attr, p=attr_drop_p)  # CFG 训练: 随机 drop 成 null\n\n    B = x0.shape[0]\n    noise = torch.randn_like(x0)\n    timesteps = torch.randint(\n        0, scheduler.config.num_train_timesteps, (B,), device=device, dtype=torch.long\n    )\n\n    # q(x_t | x_0)\n    x_t = scheduler.add_noise(x0, noise, timesteps)\n\n    # 预测噪声\n    eps_pred = model(x_t, timesteps, attr)\n    return F.mse_loss(eps_pred, noise)\n\n\n@torch.no_grad()\ndef ddpm_sample_loop(\n    model,\n    scheduler: DDPMScheduler,\n    shape: tuple,\n    device: torch.device,\n    cond_fn,  # callable: (x_t, t, eps_uncond) -> guided_eps\n):\n    """通用 DDPM 采样, guidance 由 cond_fn 提供.\n\n    cond_fn 接收 (x_t, t_int, model) -> guided_eps.\n    这样 cfg.py 和 adg.py 可以注入不同的 guidance 公式.\n    """\n    x = torch.randn(*shape, device=device)\n    scheduler.set_timesteps(scheduler.config.num_train_timesteps)\n    for t in scheduler.timesteps:\n        eps = cond_fn(x, t, model)\n        x = scheduler.step(eps, t, x).prev_sample\n    return x.clamp(-1, 1)\n\n\nif __name__ == "__main__":\n    print("ddpm.py — import-only module, 由 train.py / sample.py 调用")\n',
    'cfg.py': '"""标准 Classifier-Free Guidance (Ho & Salimans 2022).\n\nε̃(x_t, y) = ε(x_t, ∅) + w · [ε(x_t, y) - ε(x_t, ∅)]\n\n单一 guidance scale w — 这就是 ADG 想要超越的 baseline.\n"""\n\nfrom __future__ import annotations\n\nimport torch\n\nfrom model import make_null_attr\n\n\ndef cfg_cond_fn(attr: torch.Tensor, w: float):\n    """返回一个 cond_fn,用于 ddpm_sample_loop.\n\n    Args:\n        attr: (B, K) target attribute vector\n        w: guidance scale (常用 1.0 - 7.5)\n\n    Returns:\n        callable(x_t, t, model) → guided ε\n    """\n    null_attr = make_null_attr(attr.shape[0], attr.shape[1], attr.device)\n\n    def fn(x_t: torch.Tensor, t, model) -> torch.Tensor:\n        # 拼成 batch 一次过 unet (省 forward pass)\n        x_in = torch.cat([x_t, x_t], dim=0)\n        a_in = torch.cat([null_attr, attr], dim=0)\n        t_in = torch.full((x_in.shape[0],), int(t), device=x_t.device, dtype=torch.long)\n\n        eps_full = model(x_in, t_in, a_in)\n        eps_uncond, eps_cond = eps_full.chunk(2, dim=0)\n        return eps_uncond + w * (eps_cond - eps_uncond)\n\n    return fn\n\n\nif __name__ == "__main__":\n    print("cfg.py — 由 sample.py --method cfg 调用")\n',
    'adg.py': '"""Attribute-Disentangled Guidance (ADG) — 主贡献.\n\nε̃(x_t, y) = ε(x_t, ∅) + Σ_k w_k · [ε(x_t, y^(k)) - ε(x_t, ∅)]\n\n其中 y^(k) 是只激活第 k 个 attribute 的 conditioning 向量 (其他 K-1 个 = 0).\n\n每个 attribute 有独立 guidance scale w_k:\n- w_k > 0: 强化第 k 个 attribute\n- w_k = 0: 让第 k 个 attribute 自由 (跟 unconditional 一样)\n- w_k < 0: 反向 — attribute negation (proposal §5 提到的 novelty)\n"""\n\nfrom __future__ import annotations\n\nimport torch\n\nfrom model import make_null_attr\n\n\ndef make_single_attr_vectors(attr: torch.Tensor) -> list[torch.Tensor]:\n    """给定 (B, K) target 向量, 返回 K 个"只激活一个 attribute"的向量列表.\n\n    y^(k)[k] = attr[k], 其他位 = 0.\n    """\n    B, K = attr.shape\n    out = []\n    for k in range(K):\n        single = torch.zeros_like(attr)\n        single[:, k] = attr[:, k]\n        out.append(single)\n    return out\n\n\ndef adg_cond_fn(attr: torch.Tensor, ws: list[float] | torch.Tensor):\n    """ADG cond_fn for ddpm_sample_loop.\n\n    Args:\n        attr: (B, K) target attribute vector\n        ws: K-length 的 guidance scales (each w_k)\n\n    Returns:\n        callable(x_t, t, model) → guided ε\n    """\n    B, K = attr.shape\n    if not isinstance(ws, torch.Tensor):\n        ws = torch.tensor(ws, dtype=torch.float32, device=attr.device)\n    assert ws.shape == (K,), f"ws shape {ws.shape} 不匹配 K={K}"\n\n    null_attr = make_null_attr(B, K, attr.device)\n    single_attrs = make_single_attr_vectors(attr)  # K 个 (B, K)\n\n    def fn(x_t: torch.Tensor, t, model) -> torch.Tensor:\n        # 把 K+1 个 forward pass 拼成一个大 batch:\n        # [null, y^(1), y^(2), ..., y^(K)]\n        x_in = torch.cat([x_t] * (K + 1), dim=0)\n        a_in = torch.cat([null_attr] + single_attrs, dim=0)  # (B*(K+1), K)\n        t_in = torch.full((x_in.shape[0],), int(t), device=x_t.device, dtype=torch.long)\n\n        eps_full = model(x_in, t_in, a_in)  # (B*(K+1), 3, H, W)\n        eps_chunks = eps_full.chunk(K + 1, dim=0)\n        eps_uncond = eps_chunks[0]\n        eps_singles = eps_chunks[1:]  # K 个\n\n        # ADG 公式\n        guided = eps_uncond.clone()\n        for k in range(K):\n            guided = guided + ws[k] * (eps_singles[k] - eps_uncond)\n        return guided\n\n    return fn\n\n\nif __name__ == "__main__":\n    # smoke test\n    print("adg.py — smoke test")\n    attr = torch.zeros(2, 4)\n    attr[0] = torch.tensor([1.0, 1.0, 0.0, 1.0])  # smile + glasses + young\n    attr[1] = torch.tensor([0.0, 0.0, 1.0, 0.0])  # male only\n    singles = make_single_attr_vectors(attr)\n    print(f"target attr: {attr}")\n    for k, s in enumerate(singles):\n        print(f"  y^({k}): {s}")\n',
}

for fname, code in INLINED.items():
    Path(fname).write_text(code)
    print(f"  ✓ wrote {fname} ({len(code)} bytes)")

print("\nWorking dir contents:", sorted(os.listdir(".")))


## Cell 3: Adapt data.py for Kaggle CelebA path

Kaggle CelebA dataset path: `/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba/`

We need to adapt `data.py` to use this path. Create a small wrapper class.

In [ ]:
# Create kaggle_data.py — wrapper around data.py with Kaggle path
kaggle_data_code = '''
"""Kaggle-specific data wrapper for CelebA.

Kaggle CelebA dataset structure:
/kaggle/input/celeba-dataset/
  img_align_celeba/img_align_celeba/  <- images
  list_attr_celeba.csv                <- attributes

torchvision CelebA expects:
  <root>/celeba/img_align_celeba/     <- images
  <root>/celeba/list_attr_celeba.txt  <- attributes

So we need to manually point to Kaggle paths.
"""
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd

# Kaggle CelebA paths
KAGGLE_IMG_DIR = Path("/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba")
KAGGLE_ATTR_CSV = Path("/kaggle/input/celeba-dataset/list_attr_celeba.csv")

ATTR_NAMES = [
    "5_o_Clock_Shadow", "Arched_Eyebrows", "Attractive", "Bags_Under_Eyes", "Bald",
    "Bangs", "Big_Lips", "Big_Nose", "Black_Hair", "Blond_Hair", "Blurry",
    "Brown_Hair", "Bushy_Eyebrows", "Chubby", "Double_Chin", "Eyeglasses",
    "Goatee", "Gray_Hair", "Heavy_Makeup", "High_Cheekbones", "Male",
    "Mouth_Slightly_Open", "Mustache", "Narrow_Eyes", "No_Beard", "Oval_Face",
    "Pale_Skin", "Pointy_Nose", "Receding_Hairline", "Rosy_Cheeks", "Sideburns",
    "Smiling", "Straight_Hair", "Wavy_Hair", "Wearing_Earrings", "Wearing_Hat",
    "Wearing_Lipstick", "Wearing_Necklace", "Wearing_Necktie", "Young",
]
ATTR_NAME_TO_IDX = {n.lower(): i for i, n in enumerate(ATTR_NAMES)}

def attr_indices(attrs):
    return [ATTR_NAME_TO_IDX[a.lower()] for a in attrs]

class KaggleCelebASubset(Dataset):
    def __init__(self, attrs=None, resolution=64, split="train"):
        attrs = attrs or ["smiling", "eyeglasses", "male", "young"]
        self.attrs = attrs
        self.attr_idx = attr_indices(attrs)
        self.resolution = resolution
        
        # Load attribute CSV
        df = pd.read_csv(KAGGLE_ATTR_CSV)
        # CSV columns: image_id, attr1, attr2, ... (40 attrs)
        # Values: 1 or -1 → convert to 0/1
        self.image_names = df.iloc[:, 0].tolist()
        attr_cols = df.columns[1:]
        attr_matrix = df[attr_cols].values  # (N, 40)
        attr_matrix = (attr_matrix + 1) // 2  # -1→0, 1→1
        self.attr_matrix = torch.tensor(attr_matrix, dtype=torch.float32)
        
        # Split: use first 80% for train, last 20% for val (simple split)
        n = len(self.image_names)
        if split == "train":
            self.indices = list(range(int(n * 0.8)))
        else:
            self.indices = list(range(int(n * 0.8), n))
        
        self.transform = transforms.Compose([
            transforms.Resize(resolution),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ])
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        img_name = self.image_names[real_idx]
        img_path = KAGGLE_IMG_DIR / img_name
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        
        attr = self.attr_matrix[real_idx][self.attr_idx]
        return {"image": img, "attr": attr}

def get_kaggle_dataloader(attrs=None, resolution=64, batch_size=128, split="train", num_workers=2, shuffle=True):
    ds = KaggleCelebASubset(attrs=attrs, resolution=resolution, split=split)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
        pin_memory=torch.cuda.is_available(), drop_last=True,
    )
'''

with open('kaggle_data.py', 'w') as f:
    f.write(kaggle_data_code)
print("kaggle_data.py created.")

## Cell 4: Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")

In [ ]:
# 修 kaggle_data.py 的硬编码路径 bug
from pathlib import Path

base = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset")

# 自动 detect image 目录是单层还是双层嵌套
img_dir = base / "img_align_celeba"
if not list(img_dir.glob("*.jpg"))[:1] and (img_dir / "img_align_celeba").exists():
    img_dir = img_dir / "img_align_celeba"

print(f"Image dir: {img_dir}")
print(f"Sample images: {[p.name for p in sorted(img_dir.glob('*.jpg'))[:3]]}")
print(f"Attr CSV: {base / 'list_attr_celeba.csv'}")

# 替换 kaggle_data.py 的硬编码路径
content = Path("kaggle_data.py").read_text()
content = content.replace(
    '/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba',
    str(img_dir)
)
content = content.replace(
    '/kaggle/input/celeba-dataset/list_attr_celeba.csv',
    str(base / "list_attr_celeba.csv")
)
Path("kaggle_data.py").write_text(content)
print('✓ kaggle_data.py paths fixed')


## Cell 5: Smoke test data loading

In [ ]:
from kaggle_data import get_kaggle_dataloader

loader = get_kaggle_dataloader(
    attrs=["smiling", "eyeglasses", "male", "young"],
    resolution=64,
    batch_size=4,
    split="train",
    num_workers=2,
    shuffle=False,
)
print(f"Dataset size: {len(loader.dataset)}")
batch = next(iter(loader))
print(f"Image batch: {batch['image'].shape}, range [{batch['image'].min():.2f}, {batch['image'].max():.2f}]")
print(f"Attr batch: {batch['attr'].shape}, sample: {batch['attr'][0].tolist()}")
print("Data loading OK.")

## Cell 6: Training

**Time budget**: ~6h for 25 epochs on T4 (16GB VRAM, batch_size=128).

Checkpoint saved every 5 epochs to `/kaggle/working/checkpoints/`.

In [ ]:
import os
from pathlib import Path
import torch
from accelerate import Accelerator
from tqdm import tqdm

from kaggle_data import get_kaggle_dataloader
from ddpm import make_scheduler, training_step
from model import AttrConditionedUNet

# Hyperparameters (Kaggle-optimized + resume support)
ATTRS = ["smiling", "eyeglasses", "male", "young"]
RESOLUTION = 64
BATCH_SIZE = 128
EPOCHS = 15            # 总目标 15 epoch (从 epoch 5 已存起继续到 15)
LR = 2e-4
ATTR_DROP_P = 0.1
SAVE_DIR = "/kaggle/working/checkpoints"
SAVE_EVERY = 1         # 每 epoch 存一次 (防 session crash)
NUM_WORKERS = 2
MIXED_PRECISION = "fp16"
RESUME_FROM = "/kaggle/working/checkpoints/best.pt"  # 设 None 强制从头训

Path(SAVE_DIR).mkdir(exist_ok=True, parents=True)

accelerator = Accelerator(mixed_precision=MIXED_PRECISION)
device = accelerator.device
n_attrs = len(ATTRS)

# Data
loader = get_kaggle_dataloader(
    attrs=ATTRS, resolution=RESOLUTION, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, shuffle=True, split="train"
)

# Model + scheduler + optimizer
model = AttrConditionedUNet(sample_size=RESOLUTION, n_attrs=n_attrs)
scheduler = make_scheduler()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model, optimizer, loader = accelerator.prepare(model, optimizer, loader)

# Resume logic
start_epoch = 0
best_loss = float("inf")
if RESUME_FROM and os.path.exists(RESUME_FROM):
    print(f"📦 Resuming from {RESUME_FROM}...")
    state = torch.load(RESUME_FROM, map_location=device)
    accelerator.unwrap_model(model).load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    start_epoch = state["epoch"] + 1
    best_loss = state.get("best_loss", float("inf"))
    print(f"✓ Resumed at epoch {start_epoch}, best_loss so far: {best_loss:.4f}")
else:
    print("🆕 Training from scratch.")

print(f"\nDevice: {device}, mixed_precision: {MIXED_PRECISION}")
print(f"Attributes (K={n_attrs}): {ATTRS}")
print(f"Dataset size: {len(loader.dataset)}, batch size: {BATCH_SIZE}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params/1e6:.2f}M")
remaining = EPOCHS - start_epoch
print(f"Training {remaining} more epochs (epoch {start_epoch} → {EPOCHS - 1}), ~{remaining * 22} min\n")

global_step = 0
for epoch in range(start_epoch, EPOCHS):
    model.train()
    pbar = tqdm(loader, disable=not accelerator.is_main_process, desc=f"epoch {epoch}")
    epoch_losses = []
    for batch in pbar:
        optimizer.zero_grad()
        loss = training_step(model, scheduler, batch, device, attr_drop_p=ATTR_DROP_P)
        accelerator.backward(loss)
        optimizer.step()
        global_step += 1
        epoch_losses.append(loss.item())
        if global_step % 100 == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    if accelerator.is_main_process:
        avg = sum(epoch_losses) / max(len(epoch_losses), 1)
        is_best = avg < best_loss
        if is_best:
            best_loss = avg
        marker = " ⭐ best" if is_best else ""
        print(f"  epoch {epoch} done. avg loss: {avg:.4f}{marker}")

        if (epoch + 1) % SAVE_EVERY == 0 or epoch == EPOCHS - 1:
            ckpt_path = Path(SAVE_DIR) / f"ckpt_epoch{epoch + 1:03d}.pt"
            state = {
                "model": accelerator.unwrap_model(model).state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
                "best_loss": best_loss,
                "attrs": ATTRS,
                "resolution": RESOLUTION,
            }
            torch.save(state, ckpt_path)
            # best.pt = 最新 epoch (resume 用)
            torch.save(state, Path(SAVE_DIR) / "best.pt")
            # best_loss.pt = 历史最低 loss (Javen idea, sanity 用)
            if is_best:
                torch.save(state, Path(SAVE_DIR) / "best_loss.pt")
            print(f"  Saved → {ckpt_path.name}")

print(f"\nTraining done. Final best_loss: {best_loss:.4f}")
print(f"  best.pt          = 最新 epoch (resume 用)")
print(f"  best_loss.pt     = avg-loss 最低 epoch")
print(f"  ckpt_epoch*.pt   = 每 epoch 一份(防 crash)")


## Cell 7: Sampling — CFG baseline

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision
from pathlib import Path

from cfg import cfg_cond_fn
from ddpm import ddpm_sample_loop, make_scheduler
from model import AttrConditionedUNet

CKPT_PATH = "/kaggle/working/checkpoints/best.pt"
RESULT_DIR = "/kaggle/working/results"
Path(RESULT_DIR).mkdir(exist_ok=True, parents=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_attrs = 4
resolution = 64

# Load model
model = AttrConditionedUNet(sample_size=resolution, n_attrs=n_attrs).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()
scheduler = make_scheduler()

# Target: [smiling=1, eyeglasses=1, male=0, young=1]
target = torch.tensor([[1.0, 1.0, 0.0, 1.0]], device=device).repeat(16, 1)

# CFG with w=4.0
torch.manual_seed(0)
cond = cfg_cond_fn(target, w=4.0)
imgs = ddpm_sample_loop(model, scheduler, (16, 3, 64, 64), device, cond)

# Visualize
grid = torchvision.utils.make_grid(imgs, nrow=4, normalize=True, value_range=(-1, 1))
grid = grid.permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(8, 8))
plt.imshow(grid)
plt.axis("off")
plt.title("CFG (w=4.0) — target=[smiling, eyeglasses, ~male, young]", fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/cfg_baseline.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULT_DIR}/cfg_baseline.png")

## Cell 8: Sampling — ADG with per-attribute w_k

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision

from adg import adg_cond_fn
from ddpm import ddpm_sample_loop, make_scheduler

# ADG: w_smiling=1, w_eyeglasses=4 (strong), w_male=0, w_young=1
ws = [1.0, 4.0, 0.0, 1.0]
torch.manual_seed(0)
cond = adg_cond_fn(target, ws)
imgs = ddpm_sample_loop(model, scheduler, (16, 3, 64, 64), device, cond)

grid = torchvision.utils.make_grid(imgs, nrow=4, normalize=True, value_range=(-1, 1))
grid = grid.permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(8, 8))
plt.imshow(grid)
plt.axis("off")
plt.title(f"ADG (w=[1, 4, 0, 1]) — strong eyeglasses", fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/adg_strong_eyeglasses.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULT_DIR}/adg_strong_eyeglasses.png")

## Cell 9: ADG sweep — vary w_eyeglasses from 0 to 6

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision

# Sweep w_eyeglasses: 0 → 6 (7 steps), fix others at w=1
sweep_values = torch.linspace(0, 6, 7)
all_imgs = []

for v in sweep_values:
    ws = [1.0, float(v), 0.0, 1.0]  # vary w_eyeglasses
    torch.manual_seed(0)  # same seed for comparison
    cond = adg_cond_fn(target, ws)
    imgs = ddpm_sample_loop(model, scheduler, (4, 3, 64, 64), device, cond)  # 4 imgs per step
    all_imgs.append(imgs)
    print(f"w_eyeglasses={v:.1f} done")

imgs_concat = torch.cat(all_imgs, dim=0)  # (28, 3, 64, 64)
grid = torchvision.utils.make_grid(imgs_concat, nrow=4, normalize=True, value_range=(-1, 1))
grid = grid.permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(10, 12))
plt.imshow(grid)
plt.axis("off")
plt.title("ADG sweep: w_eyeglasses 0→6 (rows: w=0, 1, 2, 3, 4, 5, 6)", fontsize=10)
plt.tight_layout()
plt.savefig(f"{RESULT_DIR}/adg_sweep_eyeglasses.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULT_DIR}/adg_sweep_eyeglasses.png")

## Cell 10: Download results

Kaggle auto-saves `/kaggle/working/` to output when session ends.

Or manually download:
- `/kaggle/working/checkpoints/best.pt` (~100 MB)
- `/kaggle/working/results/*.png`

In [ ]:
import os
print("Results in /kaggle/working/results/:")
for f in os.listdir("/kaggle/working/results"):
    print(f"  {f}")
print("\nCheckpoints in /kaggle/working/checkpoints/:")
for f in os.listdir("/kaggle/working/checkpoints"):
    size = os.path.getsize(f"/kaggle/working/checkpoints/{f}") / 1e6
    print(f"  {f} ({size:.1f} MB)")

In [ ]:
   !ls -la /kaggle/working/checkpoints/
